Step 1: Set Up Hybrid Collection
Build on your previous work by creating a collection with both dense and sparse vectors:



In [ ]:
# Run this once if your environment does not already have the required packages.
# %pip install -q qdrant-client sentence-transformers python-dotenv


In [4]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import os
import time
from dotenv import load_dotenv

load_dotenv()

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

collection_name = "day3_hybrid_search"

# Recreate the collection so the notebook can be run from top to bottom multiple times.
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            index=models.SparseIndexParams(on_disk=False)
        )
    }
)


True

Step 2: Implement Dense and Sparse Encoding


In [5]:
# Dense embeddings
encoder = SentenceTransformer("all-MiniLM-L6-v2")

# A small product-support dataset designed to expose hybrid-search behavior:
# - exact identifiers such as ZX-42, ERR_CONN_RESET, CVE-2024-3094
# - semantic paraphrases such as "battery drains quickly" vs "power consumption"
# - mixed queries where both keyword precision and meaning matter
your_dataset = [
    {
        "title": "ZX-42 router firmware upgrade guide",
        "category": "networking",
        "description": "How to upgrade the ZX-42 home router to firmware 3.2.1, preserve VLAN settings, and recover after a failed update.",
    },
    {
        "title": "Troubleshooting ERR_CONN_RESET in Chrome",
        "category": "browser",
        "description": "Steps for diagnosing ERR_CONN_RESET errors, including proxy settings, DNS cache cleanup, firewall rules, and TLS inspection issues.",
    },
    {
        "title": "Laptop battery drains quickly after sleep",
        "category": "hardware",
        "description": "Identify why a notebook loses power overnight, check background wake timers, calibrate the battery, and reduce standby power consumption.",
    },
    {
        "title": "OAuth2 PKCE setup for mobile apps",
        "category": "security",
        "description": "Configure OAuth2 authorization code flow with PKCE for iOS and Android clients without storing a client secret on the device.",
    },
    {
        "title": "PostgreSQL slow query tuning checklist",
        "category": "database",
        "description": "Use EXPLAIN ANALYZE, indexes, vacuum statistics, query rewrites, and connection pooling to improve PostgreSQL response time.",
    },
    {
        "title": "Kubernetes CrashLoopBackOff debugging",
        "category": "containers",
        "description": "Investigate pods stuck in CrashLoopBackOff by checking container logs, readiness probes, missing environment variables, and image pull settings.",
    },
    {
        "title": "CVE-2024-3094 xz backdoor incident summary",
        "category": "security",
        "description": "Summary of the CVE-2024-3094 xz-utils supply-chain backdoor, affected Linux distributions, detection commands, and mitigation actions.",
    },
    {
        "title": "Resetting a forgotten admin password",
        "category": "identity",
        "description": "Recover access to an administrator account by using backup codes, identity verification, recovery email, and privileged access workflows.",
    },
    {
        "title": "Hybrid search with dense and sparse vectors",
        "category": "search",
        "description": "Combine semantic embeddings with keyword-based sparse vectors and Reciprocal Rank Fusion to improve search relevance in Qdrant.",
    },
    {
        "title": "BM25 ranking explained",
        "category": "search",
        "description": "BM25 scores documents by term frequency, inverse document frequency, and document length normalization for classic keyword retrieval.",
    },
    {
        "title": "Vector embedding similarity overview",
        "category": "search",
        "description": "Dense vector embeddings capture semantic similarity, so related concepts can match even when the exact words are different.",
    },
    {
        "title": "Docker image size reduction",
        "category": "containers",
        "description": "Shrink Docker images with multi-stage builds, smaller base images, dependency cleanup, and layer caching best practices.",
    },
    {
        "title": "API rate limit 429 handling",
        "category": "api",
        "description": "Handle HTTP 429 responses using exponential backoff, jitter, retry-after headers, request batching, and client-side throttling.",
    },
    {
        "title": "Wi-Fi roaming for warehouse scanners",
        "category": "networking",
        "description": "Tune wireless access points for barcode scanners by adjusting roaming thresholds, 802.11k/v settings, and channel overlap.",
    },
    {
        "title": "Data backup retention policy",
        "category": "operations",
        "description": "Define backup schedules, retention windows, restore testing, offsite copies, and compliance requirements for business data.",
    },
]

# Global vocabulary - automatically extends as new texts are processed
global_vocabulary = {}

# Simple sparse encoding (term-frequency style)
def create_sparse_vector(text):
    """Create sparse vector from text using term frequency."""
    from collections import Counter
    import re

    words = re.findall(r"\b\w+\b", text.lower())
    word_counts = Counter(words)

    indices = []
    values = []
    for word, count in word_counts.items():
        if word not in global_vocabulary:
            global_vocabulary[word] = len(global_vocabulary)
        indices.append(global_vocabulary[word])
        values.append(float(count))

    return models.SparseVector(indices=indices, values=values)

# Upload hybrid data. Sparse vectors live in the same named vector dictionary as dense vectors.
points = []
for i, item in enumerate(your_dataset):
    dense_vector = encoder.encode(item["description"]).tolist()
    sparse_vector = create_sparse_vector(item["description"])

    points.append(models.PointStruct(
        id=i,
        vector={"dense": dense_vector, "sparse": sparse_vector},
        payload=item
    ))

client.upsert(collection_name=collection_name, points=points)
print(f"Uploaded {len(points)} documents. Vocabulary size: {len(global_vocabulary)}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Uploaded 15 documents. Vocabulary size: 217


Step 3: Implement Hybrid Search with RRF

In [6]:
def hybrid_search_with_rrf(query_text, limit=10):
    """Perform hybrid search using Reciprocal Rank Fusion."""

    query_dense = encoder.encode(query_text).tolist()
    query_sparse = create_sparse_vector(query_text)

    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=query_dense,
                using="dense",
                limit=20
            ),
            models.Prefetch(
                query=query_sparse,
                using="sparse",
                limit=20
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit
    )

    return response.points

# Test hybrid search with a mixed semantic + exact-keyword query.
results = hybrid_search_with_rrf("how to fix ERR_CONN_RESET network error", limit=5)
for i, point in enumerate(results, 1):
    print(f"{i}. {point.payload.get('title', 'No title')} (Score: {point.score:.3f})")
    print(f"   {point.payload.get('description', '')}")


1. ZX-42 router firmware upgrade guide (Score: 0.833)
   How to upgrade the ZX-42 home router to firmware 3.2.1, preserve VLAN settings, and recover after a failed update.
2. Troubleshooting ERR_CONN_RESET in Chrome (Score: 0.667)
   Steps for diagnosing ERR_CONN_RESET errors, including proxy settings, DNS cache cleanup, firewall rules, and TLS inspection issues.
3. Resetting a forgotten admin password (Score: 0.417)
   Recover access to an administrator account by using backup codes, identity verification, recovery email, and privileged access workflows.
4. PostgreSQL slow query tuning checklist (Score: 0.410)
   Use EXPLAIN ANALYZE, indexes, vacuum statistics, query rewrites, and connection pooling to improve PostgreSQL response time.
5. Hybrid search with dense and sparse vectors (Score: 0.283)
   Combine semantic embeddings with keyword-based sparse vectors and Reciprocal Rank Fusion to improve search relevance in Qdrant.


Step 4: Compare Search Approaches

In [7]:
def compare_search_methods(query_text, limit=5):
    """Compare dense, sparse, and hybrid search results."""

    print(f"Query: '{query_text}'\n")

    dense_results = client.query_points(
        collection_name=collection_name,
        query=encoder.encode(query_text).tolist(),
        using="dense",
        limit=limit
    ).points

    sparse_results = client.query_points(
        collection_name=collection_name,
        query=create_sparse_vector(query_text),
        using="sparse",
        limit=limit
    ).points

    hybrid_results = hybrid_search_with_rrf(query_text, limit=limit)

    result_groups = {
        "DENSE SEARCH": dense_results,
        "SPARSE SEARCH": sparse_results,
        "HYBRID SEARCH (RRF)": hybrid_results,
    }

    for label, results in result_groups.items():
        print(label)
        for i, point in enumerate(results, 1):
            print(f"  {i}. {point.payload.get('title', 'No title')} ({point.score:.3f})")
        print()

    print("-" * 70)
    return result_groups

# Test with exact-keyword, semantic, and mixed query types.
test_queries = [
    "CVE-2024-3094 xz mitigation commands",              # exact identifier: sparse should help
    "my laptop loses charge while sleeping",             # semantic paraphrase: dense should help
    "Qdrant combine keyword search with embeddings RRF", # mixed concept + exact term
    "pod keeps restarting in Kubernetes",                # semantic match for CrashLoopBackOff
    "HTTP 429 retry-after exponential backoff",          # keyword-heavy API query
]

comparison_results = {}
for query in test_queries:
    comparison_results[query] = compare_search_methods(query)


Query: 'CVE-2024-3094 xz mitigation commands'

DENSE SEARCH
  1. CVE-2024-3094 xz backdoor incident summary (0.634)
  2. ZX-42 router firmware upgrade guide (0.224)
  3. API rate limit 429 handling (0.178)
  4. Data backup retention policy (0.170)
  5. Docker image size reduction (0.158)

SPARSE SEARCH
  1. CVE-2024-3094 xz backdoor incident summary (6.000)

HYBRID SEARCH (RRF)
  1. CVE-2024-3094 xz backdoor incident summary (1.000)
  2. ZX-42 router firmware upgrade guide (0.333)
  3. API rate limit 429 handling (0.250)
  4. Data backup retention policy (0.200)
  5. Docker image size reduction (0.167)

----------------------------------------------------------------------
Query: 'my laptop loses charge while sleeping'

DENSE SEARCH
  1. Laptop battery drains quickly after sleep (0.658)
  2. Troubleshooting ERR_CONN_RESET in Chrome (0.072)
  3. Kubernetes CrashLoopBackOff debugging (0.052)
  4. Hybrid search with dense and sparse vectors (0.050)
  5. Resetting a forgotten admin passwor

Step 5: Analyze Your Results
Use the outputs from Step 4 to evaluate how hybrid compares to the single approaches. Focus on when hybrid fixes dense misses (rare keywords, exact identifiers) and when sparse misses (synonyms/semantic paraphrases). Optionally, time each method (dense/sparse/hybrid) on a few queries and note average latency.


In [8]:
def timed_search(fn, query_text, runs=3):
    """Return search results and average latency in milliseconds."""
    elapsed = []
    last_results = None
    for _ in range(runs):
        start = time.perf_counter()
        last_results = fn(query_text)
        elapsed.append((time.perf_counter() - start) * 1000)
    return last_results, sum(elapsed) / len(elapsed)


def dense_search(query_text, limit=5):
    return client.query_points(
        collection_name=collection_name,
        query=encoder.encode(query_text).tolist(),
        using="dense",
        limit=limit
    ).points


def sparse_search(query_text, limit=5):
    return client.query_points(
        collection_name=collection_name,
        query=create_sparse_vector(query_text),
        using="sparse",
        limit=limit
    ).points


def titles(points):
    return [point.payload["title"] for point in points]


def reciprocal_rank(points, expected_title):
    for rank, point in enumerate(points, 1):
        if point.payload["title"] == expected_title:
            return 1 / rank
    return 0

# Expected top match for each query. This lets us compute a small teaching-oriented MRR score.
expected_titles = {
    "CVE-2024-3094 xz mitigation commands": "CVE-2024-3094 xz backdoor incident summary",
    "my laptop loses charge while sleeping": "Laptop battery drains quickly after sleep",
    "Qdrant combine keyword search with embeddings RRF": "Hybrid search with dense and sparse vectors",
    "pod keeps restarting in Kubernetes": "Kubernetes CrashLoopBackOff debugging",
    "HTTP 429 retry-after exponential backoff": "API rate limit 429 handling",
}

analysis_rows = []
for query in test_queries:
    dense_points, dense_ms = timed_search(lambda q: dense_search(q), query)
    sparse_points, sparse_ms = timed_search(lambda q: sparse_search(q), query)
    hybrid_points, hybrid_ms = timed_search(lambda q: hybrid_search_with_rrf(q, limit=5), query)

    expected = expected_titles[query]
    dense_titles = titles(dense_points)
    sparse_titles = titles(sparse_points)
    hybrid_titles = titles(hybrid_points)

    analysis_rows.append({
        "query": query,
        "expected": expected,
        "dense_top1": dense_titles[0] if dense_titles else None,
        "sparse_top1": sparse_titles[0] if sparse_titles else None,
        "hybrid_top1": hybrid_titles[0] if hybrid_titles else None,
        "dense_rr": reciprocal_rank(dense_points, expected),
        "sparse_rr": reciprocal_rank(sparse_points, expected),
        "hybrid_rr": reciprocal_rank(hybrid_points, expected),
        "dense_ms": dense_ms,
        "sparse_ms": sparse_ms,
        "hybrid_ms": hybrid_ms,
        "dense_sparse_overlap": len(set(dense_titles) & set(sparse_titles)),
        "hybrid_union_coverage": len(set(hybrid_titles) & (set(dense_titles) | set(sparse_titles))),
    })

print("Per-query analysis")
for row in analysis_rows:
    print(f"\nQuery: {row['query']}")
    print(f"  Expected: {row['expected']}")
    print(f"  Top-1 dense/sparse/hybrid: {row['dense_top1']} | {row['sparse_top1']} | {row['hybrid_top1']}")
    print(f"  Reciprocal rank dense/sparse/hybrid: {row['dense_rr']:.2f} | {row['sparse_rr']:.2f} | {row['hybrid_rr']:.2f}")
    print(f"  Avg latency ms dense/sparse/hybrid: {row['dense_ms']:.1f} | {row['sparse_ms']:.1f} | {row['hybrid_ms']:.1f}")
    print(f"  Dense/sparse top-5 overlap: {row['dense_sparse_overlap']} docs")

method_names = ["dense", "sparse", "hybrid"]
print("\nSummary")
for method in method_names:
    mrr = sum(row[f"{method}_rr"] for row in analysis_rows) / len(analysis_rows)
    avg_latency = sum(row[f"{method}_ms"] for row in analysis_rows) / len(analysis_rows)
    print(f"  {method.title():<6} MRR@5={mrr:.2f}, avg latency={avg_latency:.1f} ms")

print("\nInterpretation guide:")
print("  - Sparse search usually wins when the query contains rare exact identifiers such as CVE IDs, error codes, or HTTP status codes.")
print("  - Dense search usually helps when the query paraphrases the document, for example 'loses charge' vs 'battery drains'.")
print("  - Hybrid RRF is useful when either signal might be right; it rewards documents that rank well in both result lists.")


Per-query analysis

Query: CVE-2024-3094 xz mitigation commands
  Expected: CVE-2024-3094 xz backdoor incident summary
  Top-1 dense/sparse/hybrid: CVE-2024-3094 xz backdoor incident summary | CVE-2024-3094 xz backdoor incident summary | CVE-2024-3094 xz backdoor incident summary
  Reciprocal rank dense/sparse/hybrid: 1.00 | 1.00 | 1.00
  Avg latency ms dense/sparse/hybrid: 149.0 | 23.0 | 80.3
  Dense/sparse top-5 overlap: 1 docs

Query: my laptop loses charge while sleeping
  Expected: Laptop battery drains quickly after sleep
  Top-1 dense/sparse/hybrid: Laptop battery drains quickly after sleep | Laptop battery drains quickly after sleep | Laptop battery drains quickly after sleep
  Reciprocal rank dense/sparse/hybrid: 1.00 | 1.00 | 1.00
  Avg latency ms dense/sparse/hybrid: 83.4 | 22.3 | 84.3
  Dense/sparse top-5 overlap: 1 docs

Query: Qdrant combine keyword search with embeddings RRF
  Expected: Hybrid search with dense and sparse vectors
  Top-1 dense/sparse/hybrid: Hybrid searc